# Player-level out-of-fold switch probabilities

This notebook trains five copies of the selected nine-feature switch model. Each copy excludes one complete player fold and predicts only that held-out fold. The combined outputs give every original training example a switch probability from a model that never trained on that player.

This stage does not train outcome heads, calibrate probabilities, or make recommendations. It saves raw logits and probabilities so calibration and overlap can be evaluated next.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_FOLDER = Path("clash2")  # Relative to MyDrive.
REPOSITORY_URL = "https://github.com/jfbami/clash2.git"
MAX_EPOCHS = 60
PATIENCE = 8
MIN_DELTA = 1e-4
BATCH_SIZE = 512
MODEL_SEED = 17


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_PROJECT_FOLDER
DRIVE_CACHE = DRIVE_PROJECT_ROOT / "data/switch_current_deck_count_ablation/arrays"
OUTPUT_DIR = DRIVE_PROJECT_ROOT / "data/switch_propensity_oof"
required = [
    DRIVE_CACHE / "metadata.json",
    DRIVE_CACHE / "next_wins.npy",
    DRIVE_CACHE / "propensity_folds.npy",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required cache files:\n" + "\n".join(missing))
print(f"Source cache: {DRIVE_CACHE}")
print(f"Persistent output: {OUTPUT_DIR}")


In [ ]:
import shutil
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → T4 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CODE_ROOT = Path("/content/clash2_code")
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)

SESSION_CACHE = Path("/content/clash2_current_deck_count_arrays")
source_metadata = (DRIVE_CACHE / "metadata.json").read_bytes()
cache_is_current = (
    (SESSION_CACHE / "metadata.json").exists()
    and (SESSION_CACHE / "metadata.json").read_bytes() == source_metadata
)
if not cache_is_current:
    if SESSION_CACHE.exists():
        shutil.rmtree(SESSION_CACHE)
    shutil.copytree(DRIVE_CACHE, SESSION_CACHE)
print(f"Session cache ready: {SESSION_CACHE}")


In [ ]:
command = [
    sys.executable, "-u", str(CODE_ROOT / "scripts/train_switch_oof.py"),
    "--cache", str(SESSION_CACHE),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--min-delta", str(MIN_DELTA),
    "--batch-size", str(BATCH_SIZE),
    "--seed", str(MODEL_SEED),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)


## Inspect the raw out-of-fold probabilities

The metrics below are calculated only on original training examples. Each prediction is out of fold at the player level. The probabilities have not been recalibrated.

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads((OUTPUT_DIR / "results.json").read_text(encoding="utf-8"))
fold_rows = []
for run in report["runs"]:
    fold_rows.append({
        "fold": run["fold"],
        "heldout_examples": run["heldout_examples"],
        "best_epoch": run["best_epoch"],
        **run["heldout"],
    })
display(pd.DataFrame(fold_rows).set_index("fold").round(6))
print("Pooled out-of-fold metrics")
display(pd.Series(report["pooled_oof"], name="value").to_frame().round(6))
print("Raw probability ranges")
display(pd.Series(report["probabilities"]).to_frame(name="value"))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

probabilities = np.load(OUTPUT_DIR / "oof_switch_probabilities.npy")
labels = np.load(SESSION_CACHE / "labels.npy")
splits = np.load(SESSION_CACHE / "splits.npy")
selected = splits == 0
probabilities = probabilities[selected]
labels = labels[selected]
edges = np.linspace(0.0, 1.0, 11)
bin_ids = np.minimum(np.digitize(probabilities, edges[1:-1]), 9)
predicted = []
observed = []
counts = []
for bin_id in range(10):
    in_bin = bin_ids == bin_id
    if in_bin.any():
        predicted.append(probabilities[in_bin].mean())
        observed.append(labels[in_bin].mean())
        counts.append(in_bin.sum())
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
plt.scatter(predicted, observed, s=np.sqrt(counts) * 3, label="OOF deciles")
plt.xlabel("Predicted switch probability")
plt.ylabel("Observed switch rate")
plt.title("Raw out-of-fold calibration")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


Outputs are saved under `MyDrive/clash2/data/switch_propensity_oof`. Re-running the notebook resumes incomplete folds. Attach the executed notebook for review before calibration or outcome-head work begins.